# Notebook 02 — Preparação para Modelagem

Parte da base tratada produzida no notebook 01 e a prepara para os modelos:
separação por finalidade, divisão treino/teste, transformações e encoding. O
princípio que rege esta fase é a **prevenção de vazamento (leakage)**: a divisão
treino/teste é feita logo no início, e toda transformação que aprende algo dos
dados (estatísticas, parâmetros de encoding) será ajustada **apenas no treino**.

In [2]:
import pandas as pd

df = pd.read_parquet("data/processed/imoveis_tratados.parquet")

print("Base tratada carregada:", df.shape)
df.info()

Base tratada carregada: (12759, 16)
<class 'pandas.DataFrame'>
Index: 12759 entries, 0 to 13639
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Price             12759 non-null  int64  
 1   Condo             10872 non-null  float64
 2   Size              12759 non-null  int64  
 3   Rooms             12759 non-null  int64  
 4   Toilets           12759 non-null  int64  
 5   Suites            12759 non-null  int64  
 6   Parking           12759 non-null  int64  
 7   Elevator          12759 non-null  int64  
 8   Furnished         12759 non-null  int64  
 9   Swimming Pool     12759 non-null  int64  
 10  New               12759 non-null  int64  
 11  District          12759 non-null  str    
 12  Negotiation Type  12759 non-null  str    
 13  Property Type     12759 non-null  str    
 14  Latitude          12759 non-null  float64
 15  Longitude         12759 non-null  float64
dtypes: float64(3), int64

In [3]:
# Separa a base de dados tratada entre as finalidades de venda e locação
df_venda = df[df["Negotiation Type"] == "sale"].copy()
df_aluguel = df[df["Negotiation Type"] == "rent"].copy()

print("Venda:  ", df_venda.shape)
print("Aluguel:", df_aluguel.shape)
print("Soma:   ", len(df_venda) + len(df_aluguel))

Venda:   (6014, 16)
Aluguel: (6745, 16)
Soma:    12759


## 1. Divisão treino/teste

Antes de qualquer transformação, separamos cada base (venda e aluguel) em treino e
teste. Esta ordem é deliberada e inegociável: tudo que o modelo aprende — incluindo
estatísticas de imputação, parâmetros de encoding e o spatial lag — será calculado
**somente no treino** e aplicado ao teste. O conjunto de teste representa imóveis
"futuros", nunca vistos pelo processo de modelagem; deixá-lo influenciar qualquer
etapa do aprendizado seria vazamento (leakage), inflando artificialmente o
desempenho e mascarando como o modelo se comportaria na realidade.

Usamos divisão de 80% treino / 20% teste, com seed fixa para reprodutibilidade.

In [4]:
from sklearn.model_selection import train_test_split

# Separa as features (X) do alvo (y) em cada base.

def separar_treino_teste(dados, nome):
    X = dados.drop(columns="Price")   # tudo, menos o alvo
    y = dados["Price"]                # o alvo
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.20,      # 20% para teste
        random_state=42      # semente fixa: a divisão é sempre a mesma
    )
    print(f"{nome:8} → treino: {X_train.shape[0]:>5} | teste: {X_test.shape[0]:>5}")
    return X_train, X_test, y_train, y_test

X_train_venda, X_test_venda, y_train_venda, y_test_venda = separar_treino_teste(df_venda, "Venda")
X_train_alug,  X_test_alug,  y_train_alug,  y_test_alug  = separar_treino_teste(df_aluguel, "Aluguel")

Venda    → treino:  4811 | teste:  1203
Aluguel  → treino:  5396 | teste:  1349


Cada base foi dividida em 80% treino / 20% teste com semente fixa (`random_state=42`),
garantindo que a divisão seja idêntica a cada execução. A partir daqui, **o conjunto
de teste fica intocado** até a avaliação final dos modelos: nenhuma estatística,
transformação ou decisão de modelagem o consultará. Toda a preparação a seguir é
ajustada sobre o treino e apenas aplicada ao teste.

## 2. Definição do conjunto de variáveis (candidato de partida)

Antes de transformar qualquer coluna, fixamos quais variáveis entram no modelo, por
quê, e quais ficam de fora. Esta é a primeira de três decisões de variáveis no
projeto: aqui definimos o **conjunto candidato de partida** (o que a base já oferece);
as variáveis espaciais e socioeconômicas serão **criadas** na Fase 3; e o conjunto
**final** será decantado empiricamente na modelagem (via VIF e importância de
features). Esta decisão é estrutural e baseada em domínio — não consulta o conjunto
de teste —, portanto não viola a regra anti-leakage.

**Alvo:** `log(Price)`. A relação hedônica é multiplicativa e o preço tem forte
assimetria à direita (média ≫ mediana, vista na auditoria); o log lineariza a relação
e normaliza os resíduos, como a NBR 14653 pressupõe.

**Atributos físicos:**
- `log(Size)` — logado pelo mesmo motivo do preço: em log-log, o coeficiente vira uma
  **elasticidade** (1% a mais de área → β% a mais de preço), forma funcional correta
  do preço hedônico, com retorno marginal decrescente por m². A área também é tão
  assimétrica quanto o preço.
- `Rooms`, `Toilets`, `Suites`, `Parking` — contagens, entram diretas. Atenção
  registrada: são correlacionadas entre si (risco de multicolinearidade), a ser
  checado por VIF na modelagem.

**Comodidades / binárias:** `Elevator`, `Furnished`, `Swimming Pool`, `New`. Entram
como 0/1. Ressalva: `New` (97–99% zero) e `Furnished` têm variância baixíssima —
candidatas a sair se não contribuírem.

**Localização (baseline):** `District` → dummies (one-hot). Captura o nível médio de
preço por bairro. É a localização "grossa"; a localização fina (spatial lag,
distância, socioeconômico) será adicionada na Fase 3, permitindo medir o ganho do
tratamento espacial sobre este baseline.

**Fora do modelo:**
- `Condo` — descartado por três razões: (1) ~15% de valores ausentes; (2) risco de
  multicolinearidade com `Swimming Pool`, academia e quadra, que preferimos manter por
  estarem completas; (3) o missing é provavelmente **não-aleatório** — comodidades
  costumam ser divulgadas, mas condomínio alto em reais tende a ser omitido no
  anúncio, então o próprio `NaN` carrega viés, e imputá-lo distorceria. Aposta
  assumida: piscina/academia/quadra capturam o sinal de qualidade que o condomínio
  traria.
- `Property Type` — constante (100% apartamentos), sem poder explicativo.
- `Negotiation Type` — é o critério de separação das duas bases, não uma feature.
- `Latitude` / `Longitude` crus — não entram diretamente; são **insumo** para as
  features espaciais da Fase 3.

## 3. Transformação logarítmica de `Price` e `Size`

Aplicamos `log` ao alvo (`Price`) e à área (`Size`), pelos motivos definidos na seção
de variáveis: lineariza a relação multiplicativa do preço hedônico, transforma o
coeficiente da área em elasticidade (log-log), e normaliza a assimetria à direita de
ambas as distribuições.

O log é uma transformação **que não aprende nada dos dados** (é uma função fixa, não
um parâmetro estimado da amostra), portanto é segura aplicar a treino e teste sem
risco de vazamento — diferente da imputação e do encoding adiante. Usamos `np.log1p`
(calcula `log(1+x)`) em vez de `np.log` por robustez a eventuais zeros; o inverso,
para reverter previsões a reais, é `np.expm1`.

In [5]:
import numpy as np

# Aplica log1p ao alvo (y) de cada conjunto — treino e teste, das duas bases.
y_train_venda_log = np.log1p(y_train_venda)
y_test_venda_log  = np.log1p(y_test_venda)
y_train_alug_log  = np.log1p(y_train_alug)
y_test_alug_log   = np.log1p(y_test_alug)

# Aplica log1p à coluna Size dentro de cada X.
# Criamos uma coluna nova 'log_Size' e removemos a 'Size' original,
for X in [X_train_venda, X_test_venda, X_train_alug, X_test_alug]:
    X["log_Size"] = np.log1p(X["Size"])
    X.drop(columns="Size", inplace=True)

print("Preço de venda — original vs log (treino):")
print(f"  média/mediana original: {y_train_venda.mean():,.0f} / {y_train_venda.median():,.0f}")
print(f"  média/mediana log:      {y_train_venda_log.mean():.2f} / {y_train_venda_log.median():.2f}")

Preço de venda — original vs log (treino):
  média/mediana original: 617,926 / 390,000
  média/mediana log:      13.00 / 12.87


A transformação simetrizou as distribuições como esperado. No preço de venda
(treino), média e mediana originais distavam ~58% (R$ 617.926 vs R$ 390.000); após o
log, ficaram praticamente coladas (13,00 vs 12,87). Média ≈ mediana confirma a
simetria, validando empiricamente a decisão de modelar em log, assim a cauda à direita
diagnosticada na auditoria foi controlada. As versões em reais (`y_*`) e em log
(`y_*_log`) ficam ambas disponíveis: treina-se em log, avalia-se em reais (o MAPE é
calculado sobre valores monetários).

## 4. Encoding do `District` (one-hot)

`District` é a única variável categórica do conjunto. Transformamo-la em variáveis
dummy (one-hot): cada bairro vira uma coluna 0/1, e seu coeficiente na regressão será
o efeito médio daquele bairro sobre o preço. Escolhemos one-hot por ser a opção que
**não usa o alvo** para construir a feature — diferente do target encoding (média de
preço por bairro), que vazaria o teste no treino.

Este é o **primeiro tratamento que aprende dos dados**: o "vocabulário" de bairros é
aprendido no treino e aplicado ao teste. Por isso usamos o `OneHotEncoder` do
scikit-learn, ajustado (`fit`) só no treino — formalizando o ritual treino→teste que
até aqui era teoria.

**Limitação assumida (baseline):** um bairro que não exista no treino (raro caído só
no teste, ou novo na produção) não terá coluna. Configuramos o encoder para ignorá-lo
(`handle_unknown='ignore'`), tratando-o como bairro neutro. É frágil de propósito —
o baseline é o marco-zero comparativo, não o modelo de produto; o tratamento robusto
da localização virá pela geografia contínua (spatial lag) na Fase 3.

In [6]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

def encodar_district(X_train, X_test):
    # Cria o encoder. handle_unknown='ignore' = bairro não visto vira tudo-zero.
    # sparse_output=False faz ele devolver um array comum (mais fácil de inspecionar).
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

    # FIT só no treino: aprende o vocabulário de bairros a partir do treino.
    encoder.fit(X_train[["District"]])

    # TRANSFORM em ambos, usando o vocabulário do treino.
    train_encoded = encoder.transform(X_train[["District"]])
    test_encoded  = encoder.transform(X_test[["District"]])

    # Os nomes das novas colunas (District_Moema, District_Itaim, ...)
    nomes = encoder.get_feature_names_out(["District"])

    # Converte os arrays de volta em DataFrames, preservando o índice original
    train_df = pd.DataFrame(train_encoded, columns=nomes, index=X_train.index)
    test_df  = pd.DataFrame(test_encoded,  columns=nomes, index=X_test.index)

    # Junta as dummies ao X e remove a coluna District original de texto
    X_train_out = pd.concat([X_train.drop(columns="District"), train_df], axis=1)
    X_test_out  = pd.concat([X_test.drop(columns="District"),  test_df],  axis=1)

    print(f"Bairros no vocabulário (treino): {len(nomes)}")
    print(f"Colunas finais — treino: {X_train_out.shape[1]} | teste: {X_test_out.shape[1]}")
    return X_train_out, X_test_out

# Aplica nas duas bases
X_train_venda, X_test_venda = encodar_district(X_train_venda, X_test_venda)
X_train_alug,  X_test_alug  = encodar_district(X_train_alug,  X_test_alug)

Bairros no vocabulário (treino): 96
Colunas finais — treino: 110 | teste: 110
Bairros no vocabulário (treino): 94
Colunas finais — treino: 108 | teste: 108


O encoding produziu vocabulários de 96 bairros (venda) e 94 (aluguel), e — o ponto
crítico — treino e teste terminaram com o mesmo número de colunas em ambas as bases
(110 e 108). Essa simetria confirma que o ritual `fit`-no-treino + `handle_unknown=
'ignore'` funcionou: bairros raros caídos apenas no teste foram absorvidos como
tudo-zero, sem quebrar o alinhamento. A diferença de vocabulário entre venda (96) e
aluguel (94) mostra que as duas finalidades não cobrem exatamente os mesmos bairros, mais uma evidência a favor de modelá-las separadamente.

## 5. Limpeza final da matriz de features

Antes de modelar, removemos colunas que sobraram no `X` mas não são features:
`Negotiation Type` (já usada como critério de separação das bases) e `Property Type`
(constante — 100% apartamentos). Garantimos também que não restou nenhuma coluna de
texto, deixando a matriz inteiramente numérica, como os modelos exigem.

In [7]:
# Remove colunas que não são features
colunas_fora = ["Negotiation Type", "Property Type"]

for nome, X in [("X_train_venda", X_train_venda), ("X_test_venda", X_test_venda),
                ("X_train_alug", X_train_alug), ("X_test_alug", X_test_alug)]:
    X.drop(columns=colunas_fora, inplace=True)

# Conferência: nenhuma coluna de texto deve restar (todas numéricas)
print("Tipos de dados restantes (venda treino):")
print(X_train_venda.dtypes.value_counts())
print("\nShape final — venda treino:", X_train_venda.shape)
print("Alguma coluna de texto?", (X_train_venda.dtypes == "object").any() or (X_train_venda.dtypes == "str").any())

Tipos de dados restantes (venda treino):
float64    100
int64        8
Name: count, dtype: int64

Shape final — venda treino: (4811, 108)
Alguma coluna de texto? False
